In [ ]:
# ---------------------------------------------------------------------------
# CELL 1: SETUP DIRS
# ---------------------------------------------------------------------------
import os

ROOT_DIR = "/kaggle/working"
MODEL_DIR = os.path.join(ROOT_DIR, "qwen3tts")
VOICES_DIR = os.path.join(ROOT_DIR, "qwen3tts_voices")

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(VOICES_DIR, exist_ok=True)

In [ ]:
# ---------------------------------------------------------------------------
# CELL 2: DOWNLOAD QWEN3 BASE MODEL
# ---------------------------------------------------------------------------
from huggingface_hub import snapshot_download

print("⬇️ Downloading Qwen3-TTS 0.6B Base...")
try:
    snapshot_download(
        repo_id="Qwen/Qwen3-TTS-12Hz-0.6B-Base",
        local_dir=MODEL_DIR,
        local_dir_use_symlinks=False
    )
    print(f"✅ Model downloaded to {MODEL_DIR}")
except Exception as e:
    print(f"❌ Model download failed: {e}")

In [ ]:
# ---------------------------------------------------------------------------
# CELL 3: IMPORT VOICES FROM DATASET
# ---------------------------------------------------------------------------
import shutil
import glob

# NOTE: Kaggle dataset input paths are usually lowercase with hyphens
# Adjust 'qwen3-voices' below if your dataset name differs in the Input list
INPUT_VOICES_DIR = "/kaggle/input/qwen3-voices"

print(f"📂 Importing voices from {INPUT_VOICES_DIR}...")

if os.path.exists(INPUT_VOICES_DIR):
    # Copy wav files to working dir
    for wav in glob.glob(os.path.join(INPUT_VOICES_DIR, "*.wav")):
        shutil.copy(wav, VOICES_DIR)
        print(f"   - Copied {os.path.basename(wav)}")
else:
    print(f"⚠️ Warning: Voice dataset not found at {INPUT_VOICES_DIR}. check inputs.")

print(f"✅ Voices setup complete in {VOICES_DIR}")

In [ ]:
# ---------------------------------------------------------------------------
# CELL 4: SETUP PYAPI CODE
# ---------------------------------------------------------------------------
import sys

PYAPI_SOURCE_DIR = "/kaggle/input/sage-pyapi-code"
WORK_DIR = "/kaggle/working/pyapi"

if os.path.exists(WORK_DIR):
    shutil.rmtree(WORK_DIR)

if os.path.exists(PYAPI_SOURCE_DIR):
    shutil.copytree(PYAPI_SOURCE_DIR, WORK_DIR)
    print(f"✅ PyAPI copied to {WORK_DIR}")
else:
    print("❌ PyAPI Dataset not found!")
    sys.exit(1)

# Zrok Token
ZROK_TOKEN_PATH = "/kaggle/input/sage-zrok-token/.zrok_api_key"
zrok_token = None
if os.path.isfile(ZROK_TOKEN_PATH):
    with open(ZROK_TOKEN_PATH, "r") as f:
        zrok_token = f.read().strip()

if not zrok_token:
    print("❌ Zrok Token not found!")

In [ ]:
# ---------------------------------------------------------------------------
# CELL 5: VENV & INSTALLATION (SYSTEM PACKAGES + QWEN)
# ---------------------------------------------------------------------------
VENV_PATH = os.path.join(WORK_DIR, "venv")

# CRITICAL: Use --system-site-packages to reuse Kaggle's pre-installed PyTorch
# This avoids "no kernel image" errors caused by mismatched pip torch versions.
!python3 -m venv "{VENV_PATH}" --system-site-packages --without-pip

VENV_BIN = os.path.join(VENV_PATH, "bin")
VENV_PIP = os.path.join(VENV_BIN, "pip")

print("🔧 Bootstrapping pip...")
!wget -q https://bootstrap.pypa.io/get-pip.py -O /tmp/get-pip.py
!{os.path.join(VENV_BIN, "python")} /tmp/get-pip.py > /dev/null

print("📦 Installing Dependencies...")

# 1. Install Qwen-TTS specific deps
# We do NOT install 'torch' here, we use the system one.
!{VENV_PIP} install qwen-tts soundfile accelerate transformers --no-cache-dir > /dev/null

# 2. Install PyAPI deps
# Filter out 'torch' from the requirements file to prevent re-installation
#req_path = os.path.join(WORK_DIR, "requirements_server_cpu.txt")
#!grep -v "torch" "{req_path}" > "{req_path}.filtered"
#!{VENV_PIP} install -r "{req_path}.filtered" --no-cache-dir > /dev/null
!{VENV_PIP} install pyngrok nest_asyncio > /dev/null

print("✅ Installation complete.")

In [ ]:
import os
import time
import subprocess
import threading

# 1. OVERWRITE THE SERVICE FILE WITH T4 FIXES
# We use torch.float32 and attn_implementation="eager" to prevent T4 Crashes
code = r'''import os
import io
import uuid
import time
import torch
import logging
import soundfile as sf
from pathlib import Path
from typing import Dict, Any, Optional
from fastapi import APIRouter, HTTPException, BackgroundTasks
from fastapi.responses import Response, JSONResponse
from pydantic import BaseModel

try:
    from qwen_tts import Qwen3TTSModel
    QWEN_AVAILABLE = True
except ImportError:
    QWEN_AVAILABLE = False

logger = logging.getLogger(__name__)
router = APIRouter(tags=["qwen3tts"])

BASE_ROOT = Path(__file__).resolve().parents[2]
MODEL_DIR = BASE_ROOT / "qwen3tts" 
VOICES_DIR = BASE_ROOT / "qwen3tts_voices"

MODEL_INSTANCE = None
TASKS: Dict[str, Dict[str, Any]] = {}

class QwenCloneRequest(BaseModel):
    text: str
    ref_audio: str          
    ref_text: Optional[str] = None
    language: str = "auto" 

def load_model():
    global MODEL_INSTANCE
    if MODEL_INSTANCE is not None:
        return MODEL_INSTANCE

    if not QWEN_AVAILABLE:
        raise HTTPException(status_code=500, detail="qwen-tts package not installed.")

    logger.info(f"Loading Qwen3-TTS Model from {MODEL_DIR}...")
    
    try:
        # T4 STABILITY CONFIGURATION
        MODEL_INSTANCE = Qwen3TTSModel.from_pretrained(
            str(MODEL_DIR),
            device_map="auto",
            torch_dtype=torch.float32,     # <--- Fixes NaN/Overflow
            attn_implementation="eager"    # <--- Fixes Device-side Assert
        )
        logger.info("Qwen3-TTS Model loaded successfully (GPU/Float32/Eager).")
    except Exception as e:
        logger.error(f"Failed to load Qwen3-TTS: {e}")
        MODEL_INSTANCE = None 
        raise HTTPException(status_code=500, detail=f"Model load failed: {str(e)}")
    
    return MODEL_INSTANCE

def process_clone_task(task_id: str, text: str, ref_audio_name: str, ref_text: Optional[str], language: str):
    try:
        TASKS[task_id]["status"] = "PROCESSING"
        ref_audio_path = VOICES_DIR / ref_audio_name
        
        # Validation
        if not ref_audio_path.exists():
            raise FileNotFoundError(f"Ref audio '{ref_audio_name}' not found")

        model = load_model()

        use_x_vector = False
        if not ref_text or not ref_text.strip():
            use_x_vector = True
            logger.info(f"Task {task_id}: X-Vector Mode")
        else:
            logger.info(f"Task {task_id}: ICL Mode")

        inference_kwargs = {}
        if use_x_vector:
             inference_kwargs["x_vector_only_mode"] = True

        output, sr = model.generate_voice_clone(
            text=text,
            language=language,
            ref_audio=str(ref_audio_path),
            ref_text=ref_text,
            **inference_kwargs
        )
        
        audio_data = output[0]
        buffer = io.BytesIO()
        sf.write(buffer, audio_data, sr, format='WAV')
        buffer.seek(0)
        TASKS[task_id]["result"] = buffer.read()
        TASKS[task_id]["status"] = "COMPLETED"
        logger.info(f"Task {task_id}: Success.")

    except Exception as e:
        logger.exception(f"Task {task_id} failed.")
        TASKS[task_id]["status"] = "FAILED"
        TASKS[task_id]["error"] = str(e)

@router.get("/qwen3tts/voices")
def list_voices():
    if not VOICES_DIR.exists(): return []
    return [f.name for f in VOICES_DIR.glob("*.wav")]

@router.post("/qwen3tts/clone")
async def clone_async(req: QwenCloneRequest, background_tasks: BackgroundTasks):
    if not (VOICES_DIR / req.ref_audio).exists():
        raise HTTPException(status_code=404, detail=f"File '{req.ref_audio}' not found.")
    task_id = str(uuid.uuid4())
    TASKS[task_id] = {"status": "PENDING", "created_at": time.time()}
    background_tasks.add_task(process_clone_task, task_id, req.text, req.ref_audio, req.ref_text, req.language)
    return JSONResponse({"task_id": task_id, "status": "PENDING"})

@router.get("/qwen3tts/status/{task_id}")
async def get_task_status(task_id: str):
    task = TASKS.get(task_id)
    if not task: raise HTTPException(status_code=404, detail="Task not found")
    status = task["status"]
    if status == "COMPLETED":
        return Response(content=task["result"], media_type="audio/wav")
    elif status == "FAILED":
        err = task.get("error", "Unknown")
        del TASKS[task_id]
        return JSONResponse({"task_id": task_id, "status": "FAILED", "error": err}, status_code=500)
    else:
        return JSONResponse({"task_id": task_id, "status": status})
'''

with open("/kaggle/working/pyapi/services/qwen3tts_service.py", "w") as f:
    f.write(code)



In [ ]:
# ---------------------------------------------------------------------------
# CELL 6: ZROK
# ---------------------------------------------------------------------------
if not os.path.exists("zrok"):
    !wget https://github.com/openziti/zrok/releases/download/v1.1.3/zrok_1.1.3_linux_amd64.tar.gz > /dev/null
    !tar -xzf zrok_1.1.3_linux_amd64.tar.gz > /dev/null
    !chmod +x zrok

!./zrok disable > /dev/null 2>&1
!./zrok enable --headless "$zrok_token"

In [ ]:
# ---------------------------------------------------------------------------
# CELL 7: START SERVER
# ---------------------------------------------------------------------------
import threading
import subprocess
import time

SERVER_PORT = 8009
VENV_BIN = os.path.join(WORK_DIR, "venv", "bin")
VENV_PYTHON = os.path.join(VENV_BIN, "python")

def run_server():
    cmd = [VENV_PYTHON, "-m", "uvicorn", "main:app", "--host", "0.0.0.0", "--port", str(SERVER_PORT)]
    env = os.environ.copy()
    env["VIRTUAL_ENV"] = os.path.join(WORK_DIR, "venv")
    env["PATH"] = f"{VENV_BIN}:" + env["PATH"]
    
    print(f"🚀 Starting Qwen3-TTS API...")
    subprocess.Popen(cmd, cwd=WORK_DIR, env=env)

def start_zrok():
    time.sleep(10)
    subprocess.Popen(["./zrok", "share", "public", f"localhost:{SERVER_PORT}", "--headless"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    time.sleep(5)
    print("\n" + "="*40)
    subprocess.run(["./zrok", "overview"])
    print("="*40)

threading.Thread(target=run_server, daemon=True).start()
threading.Thread(target=start_zrok, daemon=True).start()

try:
    while True: time.sleep(60)
except KeyboardInterrupt:
    print("🛑 Stopped.")